# 원핫인코딩 (One-hot Encoding)

- 의미: 단어(또는 토큰)를 어휘 크기(vocab size)만큼의 길이를 가진 벡터로 바꾸는 방법
- 표현 방식: 해당 단어의 인덱스 위치만 1, 나머지는 전부 0 (즉, “이 단어가 맞다/아니다”만 표시)
- 예시: vocab size가 5이고 단어 인덱스가 3이면 → [0, 0, 1, 0, 0]
- 특징/주의: 단어 간 의미/유사도 정보는 없고, vocab이 커질수록 벡터가 매우 커져 메모리/연산 비용이 증가함
    - 그래서 실무에선 보통 임베딩(Embedding)으로 저차원 밀집(dense) 벡터로 바꿔 사용한다
- 사용 시기: 단어 수가 아주 작거나(카테고리/키워드 몇십~몇백) 간단한 모델/해석이 중요한 경우엔 원-핫/BoW를 사용할 때도 있다.

- 임베딩
    - 학습 가능한 변환(레이어/행렬): 단어 인덱스 k를 길이 d(예: 64, 128)인 밀집(dense) 벡터로 매핑
    - 예: k=3 → [0.12, -0.03, ...] (d차원)
    - 특징: 저차원, 밀집, 학습을 통해 의미/유사도가 반영될 수 있음

In [116]:
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

raw_text = """The Little Prince, written by Antoine de Saint-Exupéry, is a poetic tale about a young prince who travels from his home planet to Earth. The story begins with a pilot stranded in the Sahara Desert after his plane crashes. While trying to fix his plane, he meets a mysterious young boy, the Little Prince.
The Little Prince comes from a small asteroid called B-612, where he lives alone with a rose that he loves deeply. He recounts his journey to the pilot, describing his visits to several other planets. Each planet is inhabited by a different character, such as a king, a vain man, a drunkard, a businessman, a geographer, and a fox. Through these encounters, the Prince learns valuable lessons about love, responsibility, and the nature of adult behavior.
On Earth, the Little Prince meets various creatures, including a fox, who teaches him about relationships and the importance of taming, which means building ties with others. The fox's famous line, "You become responsible, forever, for what you have tamed," resonates with the Prince's feelings for his rose.
Ultimately, the Little Prince realizes that the essence of life is often invisible and can only be seen with the heart. After sharing his wisdom with the pilot, he prepares to return to his asteroid and his beloved rose. The story concludes with the pilot reflecting on the lessons learned from the Little Prince and the enduring impact of their friendship.
The narrative is a beautifully simple yet profound exploration of love, loss, and the importance of seeing beyond the surface of things."""

sentences = sent_tokenize(raw_text)

en_stopwords = stopwords.words('english')

vocab = {}

preprocessed_sentences = []

for sentence in sentences:
    sentence = sentence.lower()
    tokens = word_tokenize(sentence)
    tokens = [token for token in tokens if token not in en_stopwords]
    tokens = [token for token in tokens if len(token) > 2]

    for token in tokens:
        if token not in vocab:
            vocab[token] = 1
        else:
            vocab[token] += 1

    preprocessed_sentences.append(tokens)

print(preprocessed_sentences)

[['little', 'prince', 'written', 'antoine', 'saint-exupéry', 'poetic', 'tale', 'young', 'prince', 'travels', 'home', 'planet', 'earth'], ['story', 'begins', 'pilot', 'stranded', 'sahara', 'desert', 'plane', 'crashes'], ['trying', 'fix', 'plane', 'meets', 'mysterious', 'young', 'boy', 'little', 'prince'], ['little', 'prince', 'comes', 'small', 'asteroid', 'called', 'b-612', 'lives', 'alone', 'rose', 'loves', 'deeply'], ['recounts', 'journey', 'pilot', 'describing', 'visits', 'several', 'planets'], ['planet', 'inhabited', 'different', 'character', 'king', 'vain', 'man', 'drunkard', 'businessman', 'geographer', 'fox'], ['encounters', 'prince', 'learns', 'valuable', 'lessons', 'love', 'responsibility', 'nature', 'adult', 'behavior'], ['earth', 'little', 'prince', 'meets', 'various', 'creatures', 'including', 'fox', 'teaches', 'relationships', 'importance', 'taming', 'means', 'building', 'ties', 'others'], ['fox', 'famous', 'line', 'become', 'responsible', 'forever', 'tamed', 'resonates', '

In [117]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 상위 15개만 사용(필터링용), 그 외 토큰은 OOV 처리
tokenizer = Tokenizer(num_words=15, oov_token='<OOV>')

# 단어 빈도 기반 인덱스 사전
tokenizer.fit_on_texts(preprocessed_sentences)
sequences = tokenizer.texts_to_sequences(preprocessed_sentences)

padded = pad_sequences(sequences, maxlen=10, truncating='pre')

print(padded.shape, padded)

(13, 10) [[ 1  1  1  1  7  2  1  1  8  9]
 [ 0  0 10  1  4  1  1  1 11  1]
 [ 0  1  1 11 12  1  7  1  3  2]
 [ 1  1 13  1  1  1  1  5  1  1]
 [ 0  0  0  1  1  4  1  1  1  1]
 [ 1  1  1  1  1  1  1  1  1  6]
 [ 1  2  1  1 14  1  1  1  1  1]
 [ 1  6  1  1  1  1  1  1  1  1]
 [ 1  1  1  1  1  1  1  2  1  5]
 [ 1  3  2  1  1  1  1  1  1  1]
 [ 0  0  1  1  4  1  1 13  1  5]
 [ 1  4  1 14  1  3  2  1  1  1]
 [ 1  1  1  1  1  1  1  1  1  1]]


In [118]:
from tensorflow.keras.utils import to_categorical

one_hot_encoded = to_categorical(padded)
print(one_hot_encoded, one_hot_encoded.shape)

[[[0. 1. 0. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]
  ...
  [0. 1. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[1. 0. 0. ... 0. 0. 0.]
  [1. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 1. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]]

 [[1. 0. 0. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]
  ...
  [0. 1. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 1. ... 0. 0. 0.]]

 ...

 [[1. 0. 0. ... 0. 0. 0.]
  [1. 0. 0. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 1. 0.]
  [0. 1. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 1. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]
  ...
  [0. 1. 0. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]]

 [[0. 1. 0. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]
  ...
  [0. 1. 0. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]
  [0. 1. 0. ... 0. 0. 0.]]] (13, 10, 1

In [119]:
texts = [
    "나는 오늘 학원에 간다.",
    "친구들과 맛있는 점심 식사를 했다.",
    "오늘은 어떤 즐거운 수업을 할지 너무 기대가 된다."
]

In [120]:
from konlpy.tag import Okt
import re

okt = Okt()

In [121]:
def load_stopwords(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        stopwords = [line.strip() for line in f]
    return stopwords

ko_stopwords = load_stopwords('ko_stopwords.txt')

preprocessed_texts = []

for text in texts:
    tokens = okt.morphs(text, stem=True)
    tokens = [token for token in tokens if token not in ko_stopwords]
    tokens = [token for token in tokens if not re.search(r'[\s.,:;?!]', token)]
    preprocessed_texts.append(tokens)

print(*preprocessed_texts, sep='\n')

['는', '오늘', '학원', '간다']
['친구', '맛있다', '점심', '식사', '하다']
['오늘', '은', '어떻다', '즐겁다', '수업', '하다', '너무', '기대', '되다']


In [122]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(oov_token='<OOV')
tokenizer.fit_on_texts(preprocessed_texts)

sequences = tokenizer.texts_to_sequences(preprocessed_texts)
sequences

[[4, 2, 5, 6], [7, 8, 9, 10, 3], [2, 11, 12, 13, 14, 3, 15, 16, 17]]

In [123]:
tokenizer.word_index

{'<OOV': 1,
 '오늘': 2,
 '하다': 3,
 '는': 4,
 '학원': 5,
 '간다': 6,
 '친구': 7,
 '맛있다': 8,
 '점심': 9,
 '식사': 10,
 '은': 11,
 '어떻다': 12,
 '즐겁다': 13,
 '수업': 14,
 '너무': 15,
 '기대': 16,
 '되다': 17}

In [124]:
padded = pad_sequences(sequences, maxlen=9)

print(padded.shape, padded)

(3, 9) [[ 0  0  0  0  0  4  2  5  6]
 [ 0  0  0  0  7  8  9 10  3]
 [ 2 11 12 13 14  3 15 16 17]]


In [125]:
one_hot_encoded = to_categorical(padded)
print(one_hot_encoded, one_hot_encoded.shape)

[[[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

 [[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 

In [126]:
from tensorflow.keras import models, layers

input = layers.Input(shape=(9, 18))
x = layers.SimpleRNN(8)(input)
output = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs=input, outputs=output)
model.summary()

Model: "functional_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_17 (InputLayer)     │ (None, 9, 18)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_17 (SimpleRNN)       │ (None, 8)              │           216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225 (900.00 B)

 Trainable params: 225 (900.00 B)

 Non-trainable params: 0 (0.00 B)

In [127]:
import numpy as np

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

labels = np.array([1, 1, 1])

model.fit(one_hot_encoded, labels, epochs=10)

Epoch 1/10


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.6667 - loss: 0.8345
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6667 - loss: 0.8183
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6667 - loss: 0.8023
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6667 - loss: 0.7865
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.6667 - loss: 0.7711
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6667 - loss: 0.7558
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6667 - loss: 0.7409
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.6667 - loss: 0.7261
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6667 - loss: 0.7117
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.6667 - loss: 0.6975


임베딩 사용 전 주로 사용하던 기법인 운핫 인코딩
- vocab 차원 커질수록 메모리/연산 비효율적
- 단어 의미/유사도 정보 제대로 반영 못함

=> 정수 인덱스를 그대로 넣는 Embedding 기법 활용

- 입력 (batch, timesteps) 정수 인덱스
    -> Embedding(batch, timesteps): (batch, timesteps, vocab_size)
    -> 이후 RNN, LSTM에 넣어서 진행

| 표현 방식 | 입력 단위 | 벡터 형태(Shape) | 값이 의미하는 것 | 순서(문맥) 보존 | 장점 | 단점 | 주 사용처 |
|---|---|---|---|---|---|---|---|
| **원-핫(시퀀스)** | 토큰 시퀀스(문장) | `(seq_len, V)` 또는 `(batch, seq_len, V)` | 각 토큰을 길이 `V` 벡터로 표현하며 해당 인덱스만 1 | ✅ | 구현과 이해가 쉬움, RNN 입력으로 바로 사용 가능 | `V`가 커지면 메모리·연산량 증가, 의미·유사도 정보 없음 | 교육용 데모, 작은 vocabulary 실험 |
| **BoW (CountVectorizer)** | 문서(문장/리뷰) 1개 | `(V)` 또는 `(batch, V)` | 단어의 등장 횟수(count) | ❌ | 빠르고 간단하며 전통 ML에서 활용하기 좋음 | 단어 순서와 문맥 손실, 고차원 희소 벡터 | 스팸 분류, 감성 분석 베이스라인, 빠른 EDA |
| **TF-IDF** | 문서(문장/리뷰) 1개 | `(V)` 또는 `(batch, V)` | 단어 중요도 = TF × IDF | ❌ | BoW보다 중요한 단어를 잘 반영, 전통 ML에서 성능이 좋은 편 | 문맥과 순서 손실, 고차원 희소 벡터 | 검색, 문서 유사도, Linear SVM·로지스틱 회귀 기반 분류 |
| **Embedding** | 토큰 시퀀스(문장) | `(seq_len, d)` 또는 `(batch, seq_len, d)` | 토큰 ID를 `d`차원의 실수 벡터로 변환 | ✅ | 저차원 밀집 벡터, 의미와 유사도 표현 가능, 딥러닝에서 널리 사용 | 학습 데이터·자원 필요, 해석이 상대적으로 어려움 | RNN, LSTM, GRU, Transformer 등 딥러닝 NLP 모델 입력 |